# How the 2026 Senate and governor shifts were built, by age and sex

> **Best viewed with [nteract](https://nteract.io).** The conda environment is pinned in this notebook's metadata, so nteract can set it up for you when you open it.

**What it does**

1. Pulls the 2026 network exit poll (SSRS, via Fox News) and the 2024 baselines (Edison via CNN; AP VoteCast via Fox News where Edison didn't poll), plus the actual vote counts.
2. Shifts each year's group margins so the totals match the actual vote: the poll-vs-count gap is spread evenly across all voters.
3. Breaks each race's 2024 → 2026 swing into age × sex groups. Each group's contribution = 2026 share × margin − 2024 share × margin, so the groups sum exactly to the swing.
4. Builds an interactive Panel dashboard and writes one PNG per race to `thumbnails/`.

Downloaded pages are cached in `data/raw/`. Paths are relative to this folder.

In [1]:
import os, json, re, time, shutil, pathlib
import requests, pandas as pd
from bs4 import BeautifulSoup

# all paths are relative to this notebook's folder (set NB_DIR to run it from somewhere else)
os.chdir(os.environ.get("NB_DIR", "."))
H = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_0) AppleWebKit/537.36 Chrome/130 Safari/537.36"
}
DATA = pathlib.Path("data")
CACHE = DATA / "raw"
CACHE.mkdir(parents=True, exist_ok=True)

# reuse pages already downloaded by the age-shift notebook, if present
src = pathlib.Path("../age_shift_panel_2026/data/raw")
if src.exists():
    for p in src.iterdir():
        if not (CACHE / p.name).exists():
            shutil.copy2(p, CACHE / p.name)

STATES = {
    "AL": "alabama",
    "AK": "alaska",
    "AZ": "arizona",
    "AR": "arkansas",
    "CA": "california",
    "CO": "colorado",
    "CT": "connecticut",
    "DE": "delaware",
    "FL": "florida",
    "GA": "georgia",
    "HI": "hawaii",
    "ID": "idaho",
    "IL": "illinois",
    "IN": "indiana",
    "IA": "iowa",
    "KS": "kansas",
    "KY": "kentucky",
    "LA": "louisiana",
    "ME": "maine",
    "MD": "maryland",
    "MA": "massachusetts",
    "MI": "michigan",
    "MN": "minnesota",
    "MS": "mississippi",
    "MO": "missouri",
    "MT": "montana",
    "NE": "nebraska",
    "NV": "nevada",
    "NH": "new-hampshire",
    "NJ": "new-jersey",
    "NM": "new-mexico",
    "NY": "new-york",
    "NC": "north-carolina",
    "ND": "north-dakota",
    "OH": "ohio",
    "OK": "oklahoma",
    "OR": "oregon",
    "PA": "pennsylvania",
    "RI": "rhode-island",
    "SC": "south-carolina",
    "SD": "south-dakota",
    "TN": "tennessee",
    "TX": "texas",
    "UT": "utah",
    "VT": "vermont",
    "VA": "virginia",
    "WA": "washington",
    "WV": "west-virginia",
    "WI": "wisconsin",
    "WY": "wyoming",
}


def cached_get(url, name):
    """GET with an on-disk cache under data/raw (delete the folder to refresh)."""
    p = CACHE / name
    if p.exists():
        return p.read_text()
    r = requests.get(url, headers=H, timeout=60)
    time.sleep(0.5)
    if r.status_code != 200:
        return None
    p.write_text(r.text)
    return r.text


def devalue(arr):
    """Minimal resolver for Nuxt 3 'devalue' payloads (index-referenced JSON)."""
    memo, WRAP = (
        {},
        {
            "Reactive",
            "ShallowReactive",
            "Ref",
            "ShallowRef",
            "EmptyRef",
            "EmptyShallowRef",
            "NuxtError",
            "Set",
            "Map",
            "Date",
            "null",
        },
    )

    def res(i):
        if i in memo:
            return memo[i]
        v = arr[i]
        if isinstance(v, list) and v and isinstance(v[0], str) and v[0] in WRAP:
            out = res(v[1]) if len(v) > 1 and isinstance(v[1], int) else None
        elif isinstance(v, list):
            out = []
            memo[i] = out
            out.extend(res(x) if isinstance(x, int) else x for x in v)
            return out
        elif isinstance(v, dict):
            out = {}
            memo[i] = out
            for k, x in v.items():
                out[k] = res(x) if isinstance(x, int) else x
            return out
        else:
            out = v
        memo[i] = out
        return out

    return res(0)


def fox_questions(html):
    s = BeautifulSoup(html, "lxml").find("script", id="__NUXT_DATA__")
    root, out = devalue(json.loads(s.string)), []

    def walk(o, d=0):
        if d > 40:
            return
        if isinstance(o, dict):
            if "qcode" in o and "h_answers" in o:
                out.append(o)
            for v in o.values():
                walk(v, d + 1)
        elif isinstance(o, list):
            for v in o:
                walk(v, d + 1)

    walk(root)
    return out


def anti_gop_margin(results):
    """(top non-Republican %) - (Republican %); None when the cell is suppressed."""
    rep = opp = None
    for c in results:
        if c.get("result_percent") in (None, "", "N/A"):
            continue
        pct = float(c["result_percent"])
        if str(c.get("party", "")).upper().startswith("R"):
            rep = pct if rep is None else max(rep, pct)
        elif opp is None or pct > opp:
            opp = pct
    return None if rep is None or opp is None else opp - rep


import panel, bokeh

print(
    "setup ok",
    len(list(CACHE.iterdir())),
    "cached files · panel",
    panel.__version__,
    "bokeh",
    bokeh.__version__,
)

setup ok 202 cached files · panel 1.10.0 bokeh 3.10.0


In [2]:
# raw extraction: every (share of electorate, margin) for the age / sex questions
#   2024 -> AP VoteCast crosstabs from the Fox 2024 state pages (presidential race)
#   2026 -> SSRS Voter Poll crosstabs from Fox's voter-analysis feed, one file per Senate / governor race
URL24 = "https://www.foxnews.com/elections/2024/general-results/state/{}"
FEED = "https://feeds-elections.foxnews.com/voter_analysis"
WANT = {"age", "age45", "genderage", "genderage45", "agebygender26"}


def rows_from(questions, st, period):
    out = []
    for q in questions:
        if q.get("filter") not in (None, "All_Voters"):
            continue
        ans = [a["answer"] for a in q["h_answers"]]
        is_sex = set(ans) >= {"Male", "Female"} or set(ans) >= {"Men", "Women"}
        if q["qcode"] in WANT or is_sex:
            for a in q["h_answers"]:
                out.append(
                    dict(
                        state=st,
                        period=period,
                        qcode="SEX" if is_sex else q["qcode"],
                        answer=a["answer"],
                        share=float(a["variable_percent"])
                        if a["variable_percent"] not in (None, "")
                        else None,
                        margin=anti_gop_margin(a["results"]),
                    )
                )
    return out


raw = []
for ab, slug in STATES.items():
    html = cached_get(URL24.format(slug), f"fox_2024_{ab}.html")
    if html:
        raw += rows_from(fox_questions(html), ab, 2024)

race_list = json.loads(
    cached_get(f"{FEED}/2026_Generals/race_list.json", "va_2026_race_list.json")
)
RACES = {
    "Senate": sorted(r for r in race_list if "-S-" in r),
    "Governor": sorted(r for r in race_list if "-G-" in r),
}
for rid in RACES["Senate"] + RACES["Governor"]:
    j = json.loads(
        cached_get(f"{FEED}/2026_Generals/{rid[:2]}/{rid}.json", f"va_2026_{rid}.json")
    )
    questions = next(p for p in j if isinstance(p, list))
    raw += rows_from(questions, rid[:2], rid)
raw = pd.DataFrame(raw).drop_duplicates(["state", "period", "qcode", "answer"])
for office, rids in RACES.items():
    print(f"{office:8} {len(rids)} races:", " ".join(r[:2] for r in rids))

# cross-check: feed Senate numbers should match what the Fox state pages showed
chk = []
for rid in RACES["Senate"]:
    p = CACHE / f"fox_2026_{rid[:2]}.html"
    if not p.exists():
        continue
    page = pd.DataFrame(
        rows_from(
            [q for q in fox_questions(p.read_text()) if q["race_id"] == rid],
            rid[:2],
            rid,
        )
    )
    if page.empty:
        continue
    m = page.merge(
        raw[raw.period == rid],
        on=["state", "period", "qcode", "answer"],
        suffixes=("_page", "_feed"),
    )
    chk.append(m)
chk = pd.concat(chk)
diff = (
    (chk.share_page - chk.share_feed).abs().max(),
    (chk.margin_page - chk.margin_feed).abs().max(),
)
print(
    f"feed vs page, {len(chk)} Senate cells: max |Δshare| = {diff[0]}, max |Δmargin| = {diff[1]}"
)

Senate   35 races: AK AL AR CO DE FL GA IA ID IL KS KY LA MA ME MI MN MS MT NC NE NH NJ NM OH OK OR RI SC SD TN TX VA WV WY
Governor 36 races: AK AL AR AZ CA CO CT FL GA HI IA ID IL KS MA MD ME MI MN NE NH NM NV NY OH OK OR PA RI SC SD TN TX VT WI WY
feed vs page, 805 Senate cells: max |Δshare| = 0.0, max |Δmargin| = 0.0


In [3]:
# CNN / Edison 2024 (only states it polled): all voters + men (-20) + women (-21) subsets
cfg = requests.get(
    "https://politics.api.cnn.io/config/exit-poll-dropdown-config/2024-11-05-GE.json",
    headers=H,
    timeout=30,
).json()
ED_STATES = sorted(
    {e["stateAbbr"] for e in cfg["dropdownConfig"]["PG"] if e["subsetNumber"] == 0}
    - {"US"}
)
ED_AGE = {
    "18-29": "18-29",
    "30-44": "30-44",
    "45-64": "45-64",
    "65 or older": "65+",
    "18-44": "18-44",
    "45 or older": "45+",
}
cnn = []
for ab in ED_STATES:
    for sub, suffix in (("All", ""), ("Men", "-20"), ("Women", "-21")):
        jj = json.loads(
            cached_get(
                f"https://politics.api.cnn.io/results/exit-poll/2024-PG-{ab}{suffix}.json",
                f"cnn_2024_{ab}{suffix or '-0'}.json",
            )
        )
        for q in jj["questions"]:
            if q["question"] in ("Age", "Gender") or (
                sub == "All" and q["question"].lower().startswith("gender")
            ):
                for a in q["answers"]:
                    pc = {c["lastName"]: c["percentage"] for c in a["candidateAnswers"]}
                    ok = pc.get("Harris", -1) >= 0 and pc.get("Trump", -1) >= 0
                    cnn.append(
                        dict(
                            state=ab,
                            sub=sub,
                            question=q["question"],
                            answer=a["answer"],
                            share=a["totalPercentage"],
                            margin=(pc["Harris"] - pc["Trump"]) if ok else None,
                        )
                    )
cnn = pd.DataFrame(cnn).drop_duplicates(["state", "sub", "question", "answer"])
print(
    cnn[(cnn.sub == "All") & (cnn.question != "Age")]
    .groupby("question")
    .answer.unique()
)

Series([], Name: answer, dtype: str)


In [4]:
# ---- assemble each group's (share, margin) per state & year, filling suppressed cells from parent totals ----
AGES4 = ["18-29", "30-44", "45-64", "65+"]
FOX = {
    (r.state, r.period, r.qcode, r.answer): (r.share, r.margin)
    for r in raw.itertuples()
}
CNN = {
    (r.state, r.sub, r.question, r.answer): (
        r.share if r.share >= 0 else None,
        r.margin,
    )
    for r in cnn.itertuples()
}
nan = lambda v: v is None or v != v


def fx(st, yr, q, *answers):
    for a in answers:
        if (st, yr, q, a) in FOX:
            return FOX[(st, yr, q, a)]
    return (None, None)


def impute(kids, parents):
    """kids: {group: [share, margin, imputed]}; parents: [((share, margin), [groups])].
    If exactly one child of a parent lacks a margin (or share), back it out of the parent total."""
    changed = True
    while changed:
        changed = False
        for (ps, pm), names in parents:
            if nan(ps) or nan(pm) or any(n not in kids for n in names):
                continue
            miss_s = [n for n in names if nan(kids[n][0])]
            if len(miss_s) == 1:
                kids[miss_s[0]][0] = ps - sum(
                    kids[n][0] for n in names if n != miss_s[0]
                )
                kids[miss_s[0]][2] = True
                changed = True
            miss_m = [n for n in names if nan(kids[n][1])]
            if len(miss_m) == 1 and not any(nan(kids[n][0]) for n in names):
                n0 = miss_m[0]
                kids[n0][1] = (
                    ps * pm - sum(kids[n][0] * kids[n][1] for n in names if n != n0)
                ) / kids[n0][0]
                kids[n0][2] = True
                changed = True
    return kids


def groups(st, yr, version, baseline):
    use_cnn = yr == 2024 and baseline == "Edison 2024"
    if version == "Age":
        if use_cnn:
            g = lambda a: CNN.get((st, "All", "Age", a), (None, None))
            kids = {
                a: list(g({"65+": "65 or older"}.get(a, a))) + [False] for a in AGES4
            }
            parents = [
                (g("18-44"), ["18-29", "30-44"]),
                (g("45 or older"), ["45-64", "65+"]),
            ]
        else:
            kids = {
                a: list(fx(st, yr, "age", a, "65 or over" if a == "65+" else a))
                + [False]
                for a in AGES4
            }
            parents = [
                (fx(st, yr, "age45", "18-44"), ["18-29", "30-44"]),
                (fx(st, yr, "age45", "45+"), ["45-64", "65+"]),
            ]
    elif baseline == "VoteCast 2024":  # sex x 2 age bands (all VoteCast offers)
        names = ["Men 18-44", "Men 45+", "Women 18-44", "Women 45+"]
        if yr != 2024:  # 2026: yr is the race id
            src = {
                "Men 18-44": "Male 18-44",
                "Men 45+": "Male 45+",
                "Women 18-44": "Female 18-44",
                "Women 45+": "Female 45+",
            }
            kids = {n: list(fx(st, yr, "genderage", src[n])) + [False] for n in names}
            sexm, sexw = fx(st, yr, "SEX", "Male"), fx(st, yr, "SEX", "Female")
        else:
            src = {
                "Men 18-44": "18-44 men",
                "Men 45+": "45+ men",
                "Women 18-44": "18-44 women",
                "Women 45+": "45+ women",
            }
            kids = {n: list(fx(st, yr, "genderage45", src[n])) + [False] for n in names}
            sexm, sexw = fx(st, yr, "SEX", "Men"), fx(st, yr, "SEX", "Women")
        parents = [
            (fx(st, yr, "age45", "18-44"), ["Men 18-44", "Women 18-44"]),
            (fx(st, yr, "age45", "45+"), ["Men 45+", "Women 45+"]),
            (sexm, ["Men 18-44", "Men 45+"]),
            (sexw, ["Women 18-44", "Women 45+"]),
        ]
    else:  # Edison baseline: sex x 4 age brackets
        names = [f"{s} {a}" for s in ("Men", "Women") for a in AGES4]
        if yr != 2024:
            kids = {n: list(fx(st, yr, "agebygender26", n)) + [False] for n in names}
            sexm, sexw = fx(st, yr, "SEX", "Male"), fx(st, yr, "SEX", "Female")
            agep = {
                a: fx(st, yr, "age", a, "65 or over" if a == "65+" else a)
                for a in AGES4
            }
        else:
            sexm, sexw = (
                CNN.get((st, "All", "Gender", "Male"), (None, None)),
                CNN.get((st, "All", "Gender", "Female"), (None, None)),
            )
            kids = {}
            for s, sx, sub in (("Men", sexm, "Men"), ("Women", sexw, "Women")):
                for a in AGES4:
                    sh, m = CNN.get(
                        (st, sub, "Age", {"65+": "65 or older"}.get(a, a)), (None, None)
                    )
                    kids[f"{s} {a}"] = [
                        None if nan(sh) or nan(sx[0]) else sh * sx[0] / 100,
                        m,
                        False,
                    ]
            agep = {
                a: CNN.get(
                    (st, "All", "Age", {"65+": "65 or older"}.get(a, a)), (None, None)
                )
                for a in AGES4
            }
        parents = [
            (sexm, [f"Men {a}" for a in AGES4]),
            (sexw, [f"Women {a}" for a in AGES4]),
        ] + [(agep[a], [f"Men {a}", f"Women {a}"]) for a in AGES4]
    kids = impute(kids, parents)
    return pd.DataFrame(
        [dict(group=k, share=v[0], margin=v[1], imputed=v[2]) for k, v in kids.items()]
    )


def decompose(office, version, baseline):
    """One row per (race, group). 2024 = Harris-Trump baseline; 2026 = that state's Senate or governor race."""
    rows, dropped = [], []
    for rid in RACES[office]:
        st = rid[:2]
        if baseline == "Edison 2024" and st not in ED_STATES:
            continue
        a, b = groups(st, 2024, version, baseline), groups(st, rid, version, baseline)
        g = a.merge(b, on="group", suffixes=("24", "26"))
        if g[["share24", "margin24", "share26", "margin26"]].isna().any().any():
            dropped.append(st)
            continue
        g["w24"], g["w26"] = g.share24 / g.share24.sum(), g.share26 / g.share26.sum()
        g["shift"] = g.margin26 - g.margin24
        g["contrib"] = (
            g.w26 * g["shift"]
        )  # persuasion inside the group, weighted by its 2026 share
        mix = (
            (g.w26 - g.w24) * g.margin24
        ).sum()  # change in who turned out, valued at 2024 margins
        ov24, ov26 = (g.w24 * g.margin24).sum(), (g.w26 * g.margin26).sum()
        g["imputed"] = g.imputed24 | g.imputed26
        g = g.assign(
            state=st,
            race=rid,
            office=office,
            version=version,
            baseline=baseline,
            mix=mix,
            overall=ov26 - ov24,
            ov24=ov24,
            ov26=ov26,
        )
        assert abs(g.contrib.sum() + mix - (ov26 - ov24)) < 1e-9
        rows.append(g)
    return (pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()), dropped


parts = []
for office in ("Senate", "Governor"):
    for version in ("Age", "Age x sex"):
        for baseline in ("VoteCast 2024", "Edison 2024"):
            df, dropped = decompose(office, version, baseline)
            parts.append(df)
            print(
                f"{office:8} · {version:9} · {baseline:13} · {df.state.nunique() if len(df) else 0:2} states · "
                f"imputed cells {int(df.imputed.sum()) if len(df) else 0:2} · dropped: {dropped}"
            )
wf = pd.concat(parts, ignore_index=True)
wf.to_csv(DATA / "waterfall_senate_governor.csv", index=False)
(
    wf[(wf.version == "Age x sex") & (wf.baseline == "VoteCast 2024")]
    .groupby(["office", "state"])[["overall", "mix", "ov24", "ov26"]]
    .first()
    .round(1)
    .unstack("office")
    .swaplevel(axis=1)
    .sort_index(axis=1)
)

Senate   · Age       · VoteCast 2024 · 33 states · imputed cells 14 · dropped: ['DE', 'WY']
Senate   · Age       · Edison 2024   ·  6 states · imputed cells  0 · dropped: []
Senate   · Age x sex · VoteCast 2024 · 35 states · imputed cells 10 · dropped: []
Senate   · Age x sex · Edison 2024   ·  6 states · imputed cells  0 · dropped: []
Governor · Age       · VoteCast 2024 · 34 states · imputed cells 11 · dropped: ['HI', 'WY']
Governor · Age       · Edison 2024   ·  9 states · imputed cells  0 · dropped: []
Governor · Age x sex · VoteCast 2024 · 36 states · imputed cells 10 · dropped: []
Governor · Age x sex · Edison 2024   ·  9 states · imputed cells  3 · dropped: []


office Governor                     Senate                    
            mix  ov24  ov26 overall    mix  ov24  ov26 overall
state                                                         
AK         -4.0 -14.2  19.3    33.5   -3.8 -14.2  -0.4    13.8
AL         -2.9 -30.7  -4.6    26.1   -2.8 -30.7  -9.1    21.5
AR         -1.7 -31.3 -15.5    15.8   -1.7 -31.3 -12.1    19.2
AZ         -1.1  -6.4  11.7    18.1    NaN   NaN   NaN     NaN
CA         -0.3  19.7  28.5     8.7    NaN   NaN   NaN     NaN
CO         -0.6  10.1  29.0    19.0   -0.6  10.1  26.4    16.3
CT         -0.7  13.9  17.7     3.8    NaN   NaN   NaN     NaN
DE          NaN   NaN   NaN     NaN    1.7  14.4  26.9    12.4
FL         -1.2 -13.5  -1.7    11.8   -1.2 -13.5  -3.1    10.4
GA         -2.3  -2.5   7.7    10.2   -2.3  -2.5  17.7    20.2
HI          0.9  23.6  44.8    21.3    NaN   NaN   NaN     NaN
IA          0.2 -14.3  10.4    24.7    0.2 -14.3   3.5    17.8
ID          0.8 -36.5 -25.0    11.5    0.7 -36.5 -24.0    12.5
IL         -1.1  10.6  23.7    13.1   -1.2  10.6  23.6    12.9
KS         -1.2 -16.4  -0.4    16.0   -1.1 -16.4   4.3    20.7
KY          NaN   NaN   NaN     NaN   -0.5 -31.2 -12.2    19.0
LA          NaN   NaN   NaN     NaN   -2.1 -22.6  -2.7    19.9
MA         -1.2  24.1  26.5     2.4   -1.2  24.1  29.9     5.7
MD         -1.4  28.3  30.6     2.3    NaN   NaN   NaN     NaN
ME         -1.0   5.2  16.9    11.7   -1.0   5.2   5.4     0.2
MI         -0.1  -1.6  10.3    11.8   -0.1  -1.6   6.0     7.6
MN         -1.2   2.7  11.8     9.1   -1.2   2.7   9.3     6.6
MS          NaN   NaN   NaN     NaN    1.7 -23.1   0.8    23.9
MT          NaN   NaN   NaN     NaN   -2.4 -20.1 -10.0    10.0
NC          NaN   NaN   NaN     NaN   -2.0  -3.9  18.9    22.9
NE         -1.6 -20.3  -8.3    11.9   -1.6 -20.3  -6.3    14.0
NH         -2.2   1.9  -4.3    -6.3   -2.1   1.9  13.4    11.5
NJ          NaN   NaN   NaN     NaN   -0.3   6.0  26.9    20.9
NM          0.9   5.4  13.7     8.3    0.9   5.4  25.2    19.8
NV         -2.6  -3.7   6.7    10.4    NaN   NaN   NaN     NaN
NY         -0.4  12.9  19.4     6.5    NaN   NaN   NaN     NaN
OH         -0.4 -11.4   7.6    19.0   -0.4 -11.4   4.5    15.9
OK          0.1 -34.3  -7.3    26.9    0.1 -34.3 -19.2    15.1
OR         -0.2  12.6   6.3    -6.3   -0.2  12.6  16.0     3.4
PA         -1.0  -1.8  22.1    23.9    NaN   NaN   NaN     NaN
RI         -2.3  14.5  43.8    29.3   -2.3  14.5  36.7    22.3
SC         -0.7 -18.7  -4.1    14.6   -0.7 -18.7   4.9    23.6
SD         -0.1 -30.2 -26.7     3.5   -0.1 -30.2 -21.2     9.0
TN         -1.8 -29.9  -2.5    27.4   -1.8 -29.9 -12.0    17.9
TX         -0.9 -14.0   6.9    21.0   -0.9 -14.0  11.8    25.8
VA          NaN   NaN   NaN     NaN   -1.1   4.9  20.9    16.0
VT         -1.0  30.7  -8.4   -39.1    NaN   NaN   NaN     NaN
WI         -0.7  -1.5   6.6     8.2    NaN   NaN   NaN     NaN
WV          NaN   NaN   NaN     NaN   -0.5 -43.1 -20.4    22.7
WY         -1.4 -46.6 -46.9    -0.3   -1.4 -46.6 -30.0    16.7

## Dashboard — how the Senate / governor shift per state was built, by age group
**2026 race** switches between the 35 Senate and 36 governor races; tabs switch between **age only** and **age × sex**; the other buttons pick the 2024 baseline, sort order and panel size. Hover any bar for the group's share, both margins and its step.

Each waterfall: **bar width** = the group's share of 2026 voters; **bar height** = points that group added to the D−R margin (share × its own shift). Steps run young → old (men solid, women hatched), then a **turnout-mix** step, ending at the state's **overall** shift — the pieces sum exactly: Σ w₂₆·(m₂₆ − m₂₄) + Σ (w₂₆ − w₂₄)·m₂₄ = overall. The baseline for both offices is the 2024 Harris–Trump vote, so governor races with strong incumbents (e.g. VT) show big candidate effects. Suppressed small cells are backed out of group totals and drawn with a dashed outline.

**Sources.** 2026: SSRS Voter Poll via Fox's voter-analysis feed (`feeds-elections.foxnews.com/voter_analysis/2026_Generals/<ST>/<race>.json`; Senate numbers checked cell-for-cell against the Fox state pages). 2024: AP VoteCast (Fox 2024 state pages) or the Edison exit poll (CNN API).

In [5]:
import numpy as np
import panel as pn
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool, Range1d, Span, Label

pn.extension()

# ---------------- encoding ----------------
# x: each group's share of the 2026 electorate (bars laid end to end across 0–100%)
# y: running total of points added to the D-minus-R margin; each step = share × that group's own shift,
#    then the turnout-mix step, ending at the state's overall 2024 → 2026 shift.
BLUE, RED = "#2166ac", "#b2182b"
BLUE_DK, RED_DK = "#08306b", "#67001f"
INK, MUTED, GRID = "#1f2328", "#57606a", "#d0d7de"
GAP, MIX_W, OVR_W = 4, 7, 12  # x-units after the 100% of groups
XMAX = 100 + GAP + MIX_W + GAP + OVR_W


def group_order(groups):
    """Young to old; within an age band men then women, so M and W sit side by side."""

    def key(g):
        sex, age = g.split(" ", 1) if " " in g else ("", g)
        return (
            ["18-29", "18-44", "30-44", "45-64", "45+", "65+"].index(age),
            {"Men": 0, "Women": 1}.get(sex, 0),
        )

    return sorted(groups, key=key)


fmt = lambda v: "0" if round(v) == 0 else f"{'D' if v > 0 else 'R'}+{abs(v):.0f}"
sgn = lambda v: f"{v:+.1f}"


def state_frame(d):
    """Bars for one state, in plotting order, with running totals."""
    d = d.set_index("group").loc[group_order(d.group.tolist())].reset_index()
    x0 = np.concatenate([[0], np.cumsum(d.w26.values * 100)[:-1]])
    x1 = x0 + d.w26.values * 100
    c1 = np.cumsum(d.contrib.values)
    c0 = c1 - d.contrib.values
    women = d.group.str.startswith("Women").values
    bars = dict(
        left=x0,
        right=x1,
        bottom=np.minimum(c0, c1),
        top=np.maximum(c0, c1),
        level_in=c0,
        level_out=c1,
        color=np.where(d.contrib.values >= 0, BLUE, RED),
        alpha=np.full(len(d), 0.9),
        hatch=np.where(women, "/", " "),
        line_dash=np.where(d.imputed.values, "dashed", "solid"),
        line_color=np.where(d.imputed.values, INK, "white"),
        name=d.group.values,
        share=[f"{v * 100:.0f}%" for v in d.w26],
        m24=[fmt(v) for v in d.margin24],
        m26=[fmt(v) for v in d.margin26],
        shift=[f"{v:+.0f}" for v in d["shift"]],
        step=[sgn(v) for v in d.contrib],
        note=["imputed from group totals" if v else "" for v in d.imputed],
    )
    mix, ov = d["mix"].iloc[0], d["overall"].iloc[0]
    mx0 = 100 + GAP
    ox0 = mx0 + MIX_W + GAP
    extra = dict(
        left=[mx0, ox0],
        right=[mx0 + MIX_W, ox0 + OVR_W],
        bottom=[min(c1[-1], c1[-1] + mix), min(0, ov)],
        top=[max(c1[-1], c1[-1] + mix), max(0, ov)],
        level_in=[c1[-1], 0],
        level_out=[c1[-1] + mix, ov],
        color=[BLUE if mix >= 0 else RED, BLUE_DK if ov >= 0 else RED_DK],
        alpha=[0.45, 1.0],
        hatch=[".", " "],
        line_dash=["solid", "solid"],
        line_color=["white", "white"],
        name=["Turnout mix", "Overall"],
        share=["—", "100%"],
        m24=["—", fmt(d.ov24.iloc[0])],
        m26=["—", fmt(d.ov26.iloc[0])],
        shift=["—", f"{ov:+.0f}"],
        step=[sgn(mix), sgn(ov)],
        note=[
            "change in who voted (2026 vs 2024 shares), valued at 2024 margins",
            "sum of every step",
        ],
    )
    return {k: list(bars[k]) + list(extra[k]) for k in bars}, ov


def state_fig(st, d, y_range, size):
    data, ov = state_frame(d)
    w, h = size
    p = figure(
        width=w,
        height=h,
        x_range=Range1d(-1, XMAX + 1),
        y_range=Range1d(*y_range),  # same limits everywhere, separate model per panel
        toolbar_location=None,
        tools="",
        min_border_left=34,
        min_border_right=6,
        min_border_top=4,
        min_border_bottom=16,
    )
    src = ColumnDataSource(data)
    r = p.quad(
        left="left",
        right="right",
        bottom="bottom",
        top="top",
        source=src,
        fill_color="color",
        fill_alpha="alpha",
        hatch_pattern="hatch",
        hatch_color="white",
        hatch_alpha=0.75,
        hatch_scale=6,
        line_color="line_color",
        line_dash="line_dash",
        line_width=1,
    )
    # connectors: level carried from the end of one bar to the start of the next
    xs, ys = [], []
    for i in range(len(data["left"]) - 1):
        if data["name"][i + 1] == "Overall":
            continue
        xs.append([data["right"][i], data["left"][i + 1]])
        ys.append([data["level_out"][i]] * 2)
    p.multi_line(xs, ys, line_color=MUTED, line_width=0.8, line_dash="dotted")
    p.add_layout(Span(location=0, dimension="width", line_color=INK, line_width=0.8))
    p.add_tools(
        HoverTool(
            renderers=[r],
            tooltips=f"""
        <div style='font-size:12px'><b>{st} · @name</b><br>
        share of 2026 voters: @share<br>margin: @m24 (2024) → @m26 (2026), shift @shift<br>
        <b>step: @step pts</b> <span style='color:{MUTED}'>@note</span></div>""",
        )
    )
    p.add_layout(
        Label(
            x=XMAX - OVR_W / 2,
            y=ov,
            text=f"{ov:+.0f}",
            text_align="center",
            y_offset=4 if ov >= 0 else -14,
            text_font_size="10px",
            text_font_style="bold",
            text_color=BLUE_DK if ov >= 0 else RED_DK,
        )
    )
    p.title.text = f"{st}   {fmt(d.ov24.iloc[0])} → {fmt(d.ov26.iloc[0])}"
    p.title.text_font_size = "12px"
    p.title.text_color = INK
    # x labels: one per age band (centred under the M+W pair), then the mix and overall bars
    bands = {}
    for name, l, r_ in zip(data["name"], data["left"], data["right"]):
        if name in ("Turnout mix", "Overall"):
            continue
        age = name.split(" ", 1)[-1]
        bands.setdefault(age, [l, r_])
        bands[age][1] = r_
    ticks = {
        round((l + r_) / 2, 3): a.replace("-", "–") for a, (l, r_) in bands.items()
    }
    ticks[round(100 + GAP + MIX_W / 2, 3)] = "mix"
    ticks[round(XMAX - OVR_W / 2, 3)] = "all"
    p.xaxis.ticker = list(ticks)
    p.xaxis.major_label_overrides = ticks
    p.xaxis.major_label_text_font_size = "8px" if size[0] < 300 else "9px"
    p.xaxis.major_label_standoff = 2
    p.yaxis.major_label_text_font_size = "8px"
    p.xgrid.grid_line_color = None
    p.ygrid.grid_line_color = GRID
    p.ygrid.grid_line_alpha = 0.6
    p.outline_line_color = None
    p.axis.axis_line_color = GRID
    p.axis.major_tick_line_color = GRID
    p.axis.minor_tick_line_color = None
    return p


# ---------------- controls (button groups only) ----------------
def toggle(options, value):
    return pn.widgets.RadioButtonGroup(
        options=options,
        value=value,
        button_type="primary",
        button_style="outline",
        margin=(0, 5),
    )


def label(text):
    return pn.pane.HTML(
        f"<div style='font-size:12px;font-weight:600;color:{MUTED};text-transform:uppercase;"
        f"letter-spacing:.04em;margin-bottom:2px'>{text}</div>",
        margin=(0, 5),
    )


SORTS = ["Overall shift", "2026 margin", "2024 margin", "A–Z"]
w_base = toggle(["VoteCast 2024", "Edison 2024"], "VoteCast 2024")
w_sort = toggle(SORTS, SORTS[0])
w_size = toggle(["Small", "Large"], "Small")
w_office = toggle(["Senate", "Governor"], "Senate")


def title(office):
    noun = "Senate" if office == "Senate" else "governor"
    return pn.pane.HTML(
        f"<h2 style='color:{INK};margin:0 0 4px 0'>How the {noun} shift per state was built, by age group</h2>"
        f"<div style='color:{MUTED};font-size:13px;margin-bottom:6px'>2024 presidential vote → 2026 {noun} vote</div>",
        sizing_mode="stretch_width",
    )


def legend(version, baseline):
    sw = lambda c, txt, hatch=False, alpha=1: (
        f"<span style='display:inline-flex;align-items:center;margin-right:14px'>"
        f"<span style='width:14px;height:12px;background:{c};opacity:{alpha};margin-right:5px;"
        + (
            "background-image:repeating-linear-gradient(135deg,transparent 0 3px,rgba(255,255,255,.8) 3px 5px);"
            if hatch
            else ""
        )
        + f"'></span>{txt}</span>"
    )
    if version == "Age":
        order = "18–29 → 30–44 → 45–64 → 65+"
    elif baseline == "VoteCast 2024":
        order = "Men 18–44 · Women 18–44 → Men 45+ · Women 45+"
    else:
        order = "Men · Women 18–29 → 30–44 → 45–64 → 65+"
    keys = sw(BLUE, "moved toward D") + sw(RED, "moved toward R")
    if version != "Age":
        keys += sw("#6e7781", "solid = men") + sw(
            "#6e7781", "hatched = women", hatch=True
        )
    keys += sw("#6e7781", "turnout mix", alpha=0.45) + sw(BLUE_DK, "overall")
    return pn.pane.HTML(
        f"<div style='color:{INK};font-size:12.5px;line-height:1.6'>{keys}<br>"
        f"<span style='color:{MUTED}'>Left to right: <b style='color:{INK}'>{order}</b>, then turnout mix, then overall. "
        f"Bar width = share of 2026 voters · bar height = points that group added to the margin "
        f"(share × its own shift) · dashed outline = cell filled in from group totals. Hover any bar for details.</span></div>",
        sizing_mode="stretch_width",
    )


def grid(version, baseline, how, size, office):
    d = wf[(wf.office == office) & (wf.version == version) & (wf.baseline == baseline)]
    if d.empty:
        return pn.pane.HTML(f"<p style='color:{MUTED}'>No data.</p>")
    per = d.groupby("state").agg(
        overall=("overall", "first"), ov24=("ov24", "first"), ov26=("ov26", "first")
    )
    order = {
        "Overall shift": per.overall.sort_values(ascending=False).index,
        "2026 margin": per.ov26.sort_values(ascending=False).index,
        "2024 margin": per.ov24.sort_values(ascending=False).index,
        "A–Z": sorted(per.index),
    }[how]
    # one shared y scale so heights compare across states
    lo, hi = [], []
    for st, g in d.groupby("state"):
        data, _ = state_frame(g)
        lo.append(min(data["bottom"]))
        hi.append(max(data["top"]))
    y_range = (min(min(lo), 0) - 3, max(hi) + 4)
    dims = (250, 190) if size == "Small" else (360, 260)
    figs = [
        pn.pane.Bokeh(state_fig(st, d[d.state == st], y_range, dims), margin=(4, 6))
        for st in order
    ]
    head = pn.pane.HTML(
        f"<div style='color:{INK};font-size:13px'>{len(figs)} {office.lower()} races · 2024 baseline: {baseline} · "
        f"median overall shift <b>{per.overall.median():+.1f}</b> pts · y-axis is the same in every panel</div>",
        sizing_mode="stretch_width",
    )
    return pn.Column(
        head,
        pn.FlexBox(*figs, flex_wrap="wrap", sizing_mode="stretch_width"),
        sizing_mode="stretch_width",
    )


def view(version):
    return pn.Column(
        pn.bind(legend, version, w_base),
        pn.bind(grid, version, w_base, w_sort, w_size, w_office),
        sizing_mode="stretch_width",
    )


controls = pn.FlexBox(
    pn.Column(label("2026 race"), w_office, margin=(0, 18, 10, 0)),
    pn.Column(label("2024 baseline"), w_base, margin=(0, 18, 10, 0)),
    pn.Column(label("Sort states by"), w_sort, margin=(0, 18, 10, 0)),
    pn.Column(label("Panel size"), w_size, margin=(0, 18, 10, 0)),
    flex_wrap="wrap",
    sizing_mode="stretch_width",
)

note = pn.pane.HTML(
    f"<div style='color:{MUTED};font-size:12px;line-height:1.45'>Margin = top non-Republican minus Republican. 2026 = SSRS "
    "Voter Poll, the state's Senate or governor race. 2024 = Harris–Trump from <b>AP VoteCast</b> (every state; by sex it only splits 18–44 / 45+) "
    "or the <b>Edison exit poll</b> (the states it covered: 6 with Senate races, 9 with governor races; full age × sex). Steps: each group's shift weighted by "
    "its 2026 share, plus a turnout-mix step = Σ (2026 share − 2024 share) × 2024 margin; together they sum exactly to the "
    "overall shift. Different survey methods across years: read small steps (≲2 pts) as noise.</div>",
    sizing_mode="stretch_width",
)

dashboard = pn.Column(
    pn.bind(title, w_office),
    controls,
    pn.Tabs(
        ("Age only", view("Age")),
        ("Age × sex", view("Age x sex")),
        sizing_mode="stretch_width",
        dynamic=False,
    ),
    note,
    styles={
        "background": "#ffffff",
        "color": INK,
        "border-radius": "10px",
        "padding": "18px 22px",
    },
    sizing_mode="stretch_width",
    max_width=1600,
)
dashboard.servable()

Column(max_width=1600, sizing_mode='stretch_width', styles={'background': '#ffffff', ...})
    [0] ParamFunction(function, _pane=HTML, defer_load=False)
    [1] FlexBox(objects=[Column, Column, ...], sizing_mode='stretch_width')
    [2] ...

In [19]:
# ---- actual vote results (Fox results pages, fetched fresh): 2026 race margin vs 2024 presidential margin ----
# margin = top non-Republican candidate minus Republican, as % of all votes cast in that race (same rule as the exit polls)
def nuxt_html(html):
    return devalue(
        json.loads(
            re.search(r'id="__NUXT_DATA__"[^>]*>(.*?)</script>', html, re.S).group(1)
        )
    )


def race_tables(root, depth=0):
    if depth > 30:
        return None
    if isinstance(root, dict):
        if isinstance(root.get("raceTable"), dict):
            return root["raceTable"]
        vals = root.values()
    elif isinstance(root, list):
        vals = root
    else:
        return None
    for v in vals:
        r = race_tables(v, depth + 1)
        if r is not None:
            return r


def actual_margin(state_block):
    races = state_block if isinstance(state_block, list) else [state_block]
    races = [
        r
        for r in races
        if isinstance(r, dict) and r.get("results") and not r.get("isRunoff")
    ]
    rows = [
        (r.get("partyCode"), (r.get("votes") or {}).get("count") or 0)
        for r in races[0]["results"]
        if isinstance(r, dict)
    ]
    tot = sum(c for _, c in rows)
    rep = max((c for p, c in rows if p == "R"), default=0)
    oth = max((c for p, c in rows if p != "R"), default=0)
    return 100 * (oth - rep) / tot, tot, races[0].get("last_updated")


# thumbnail races: every Edison-2024 race (age x sex, 8 groups) + Mississippi Senate on the AP VoteCast 2024 baseline (4 groups)
EXTRA_VOTECAST = [("Senate", "MS")]
ax_ = wf[wf.version == "Age x sex"]
thumb_sel = pd.concat(
    [ax_[ax_.baseline == "Edison 2024"]]
    + [
        ax_[(ax_.baseline == "VoteCast 2024") & (ax_.office == o) & (ax_.state == s)]
        for o, s in EXTRA_VOTECAST
    ]
)
NAMES = {
    "AZ": "arizona",
    "FL": "florida",
    "GA": "georgia",
    "MI": "michigan",
    "MS": "mississippi",
    "NC": "north-carolina",
    "NV": "nevada",
    "OH": "ohio",
    "PA": "pennsylvania",
    "TX": "texas",
    "WI": "wisconsin",
}
races = thumb_sel[["office", "state", "baseline", "ov24", "ov26"]].drop_duplicates(
    ["office", "state"]
)
rows = []
for st in sorted(races.state.unique()):
    h26 = requests.get(
        f"https://www.foxnews.com/elections/2026/midterm-results/state/{NAMES[st]}",
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0"},
    ).text
    pathlib.Path(f"data/raw/fox_results_2026_{st}.html").write_text(h26)
    rt26 = race_tables(nuxt_html(h26))
    p24, _, _ = actual_margin(
        race_tables(nuxt_html(open(f"data/raw/fox_2024_{st}.html").read()))[
            "president"
        ]["state"]
    )
    for office in races[races.state == st].office:
        m26, t26, upd = actual_margin(rt26[office.lower()]["state"])
        rows.append(
            dict(
                office=office,
                state=st,
                actual24=p24,
                actual26=m26,
                actual_shift=m26 - p24,
                votes26=t26,
                updated26=upd,
            )
        )
actual = pd.DataFrame(rows).merge(races, on=["office", "state"])
actual["exit_shift"] = actual.ov26 - actual.ov24
actual.to_csv("data/actual_results.csv", index=False)
actual.round(2).sort_values(["office", "actual_shift"], ascending=[False, False])

,office,state,actual24,actual26,actual_shift,votes26,updated26,baseline,ov24,ov26,exit_shift
13,Senate,TX,-13.69,12.10,25.79,9886768,2026-11-06T22:51:54.452000Z,Edison 2024,-13.56,11.82,25.37
7,Senate,MS,-22.89,0.27,23.16,953314,2026-11-06T23:02:24.400000Z,VoteCast 2024,-23.09,0.82,23.91
8,Senate,NC,-3.22,19.62,22.85,4203218,2026-11-06T20:18:39.070000Z,Edison 2024,-3.02,19.25,22.27
3,Senate,GA,-2.19,18.04,20.24,4178866,2026-11-07T01:13:13.995000Z,Edison 2024,-2.51,17.41,19.92
10,Senate,OH,-11.21,4.85,16.07,4601571,2026-11-06T21:33:13.953000Z,Edison 2024,-11.01,4.77,15.78
1,Senate,FL,-13.10,-2.88,10.22,8276945,2026-11-06T23:12:37.681000Z,Edison 2024,-12.84,-3.16,9.68
5,Senate,MI,-1.41,7.08,8.50,4796991,2026-11-06T20:52:52.553000Z,Edison 2024,-1.12,6.35,7.47
12,Governor,PA,-1.71,22.14,23.85,4985768,2026-11-07T00:06:50.056000Z,Edison 2024,-2.13,22.41,24.54
14,Governor,TX,-13.69,7.13,20.83,9905493,2026-11-06T22:51:52.822000Z,Edison 2024,-13.56,7.21,20.77
11,Governor,OH,-11.21,7.93,19.14,4595782,2026-11-06T21:33:13.939000Z,Edison 2024,-11.01,7.95,18.96


In [25]:
# ---- X-thread thumbnails (age x sex), one PNG per race -> thumbnails/ ----
# Group margins are shifted so each year's totals match the actual vote (poll-vs-count gap spread evenly across voters).
# Bar height = each group's full contribution: 2026 share x margin - 2024 share x margin (bars sum to the swing).
# Green line = EVEN (2024 margin erased); above it = 2026 margin. Uses `thumb_sel` + `actual` from the cell above.
import logging, numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import matplotlib.patheffects as pe

plt.rcParams.update(
    {
        "font.family": ["Inter", "Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
        "hatch.linewidth": 1.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
T_BLUE, T_RED, T_BLUE_DK, T_RED_DK = "#2166ac", "#b2182b", "#08306b", "#67001f"
T_GREEN = "#00c853"
T_LIGHT = "#4f8fcf"  # 2024 share labels
T_INK, T_MUTED, T_GRID, T_GREY = "#1f2328", "#57606a", "#d8dee4", "#6e7781"
T_GAP, T_OVR_W = 4, 12
T_XMAX = 100 + T_GAP + T_OVR_W + 10
T_AGES = ["18-29", "30-44", "45-64", "65+", "18-44", "45+"]

tfmt = lambda v: "even" if round(v) == 0 else f"{'D' if v > 0 else 'R'}+{abs(v):.0f}"
tfmt1 = lambda v: (
    "even" if round(v, 1) == 0 else f"{'D' if v > 0 else 'R'}+{abs(v):.1f}"
)
T_HALO = [pe.withStroke(linewidth=2.5, foreground="white")]
T_PT_PER_X = (
    0.86 * 8 * 72 / (T_XMAX + 2)
)  # points per x data unit (axes span T_XMAX+2 units)
t_text_pt = lambda t, fs: (
    len(t) * fs * 0.66
)  # rough text width in points (slightly generous)
tpct = lambda v: f"{v:.0f}%" if abs(v - round(v)) < 0.05 else f"{v:.1f}%"


def t_order(d):
    key = lambda g: (T_AGES.index(g.split(" ", 1)[1]), 0 if g.startswith("Men") else 1)
    return d.set_index("group").loc[sorted(d.group, key=key)].reset_index()


def make_thumb(
    d, office, state, path, actual=None, source_2024="Edison exit poll (via CNN)"
):
    d = t_order(d)
    d = d.assign(w24=d.w24 * 100, w26=d.w26 * 100)  # wf stores weights as fractions
    ov24, ov26 = d.ov24.iloc[0], d.ov26.iloc[0]
    if (
        actual is not None
    ):  # spread each year's poll-vs-count gap evenly across all voters
        d = d.assign(
            margin24=d.margin24 + (actual["actual24"] - ov24),
            margin26=d.margin26 + (actual["actual26"] - ov26),
        )
        ov24, ov26 = actual["actual24"], actual["actual26"]
    w = d.w26.values  # % of 2026 electorate (normalised)
    w24p = d.w24.values / d.w24.values.sum() * 100  # % of 2024 electorate (normalised)
    shift = (d.margin26 - d.margin24).values  # each group's own swing
    # full contribution: change in the group's net vote as a share of all voters
    step = (w * d.margin26.values - w24p * d.margin24.values) / 100
    x0 = np.concatenate([[0], np.cumsum(w)[:-1]])
    xc = x0 + w / 2
    c1 = np.cumsum(step)
    c0 = c1 - step
    ov = ov26 - ov24
    assert abs(c1[-1] - ov) < 0.05, (state, office, c1[-1], ov)
    women = d.group.str.startswith("Women").values
    off = office.title()
    noun = "Senate" if off == "Senate" else "governor"

    fig = plt.figure(figsize=(8, 8.9), dpi=200)
    fig.patch.set_facecolor("white")

    # ---------- header ----------
    fig.text(
        0.06,
        0.955,
        f"{state} 2026 {off} Shift",
        fontsize=26,
        fontweight="bold",
        color=T_INK,
        va="top",
    )
    direction = "toward Democrats" if ov >= 0 else "toward Republicans"
    art = "an" if str(int(abs(ov))).startswith(("8", "11", "18")) else "a"
    fig.text(
        0.06,
        0.905,
        f"2024 president {tfmt1(ov24)}  →  2026 {noun} {tfmt1(ov26)}:  "
        f"{art} {abs(ov):.1f}-point swing {direction}",
        fontsize=12,
        color=T_INK,
        va="top",
    )
    fig.text(
        0.06,
        0.875,
        "How each age × sex group added to it: width = share of 2026 voters, height = net points added",
        fontsize=10.5,
        color=T_MUTED,
        va="top",
    )
    handles = [
        Patch(facecolor=T_BLUE, label="added to D margin"),
        Patch(facecolor=T_RED, label="added to R margin"),
        Patch(facecolor=T_GREY, label="men"),
        Patch(facecolor=T_GREY, hatch="///", edgecolor="white", label="women"),
        Patch(facecolor=T_BLUE_DK, label="overall"),
    ]
    fig.legend(
        handles=handles,
        loc="upper left",
        bbox_to_anchor=(0.05, 0.858),
        ncol=5,
        frameon=False,
        fontsize=9.5,
        handlelength=1.4,
        handleheight=1.1,
        columnspacing=1.3,
        handletextpad=0.5,
    )

    # ---------- waterfall ----------
    ax = fig.add_axes([0.10, 0.455, 0.86, 0.355])
    for i in range(len(d)):
        col = T_BLUE if step[i] >= 0 else T_RED
        ax.bar(
            x0[i],
            step[i],
            width=w[i],
            bottom=c0[i],
            align="edge",
            color=col,
            alpha=0.92,
            hatch="///" if women[i] else None,
            edgecolor="white",
            linewidth=0.8,
            linestyle="--" if d.imputed.iloc[i] else "-",
        )
        if abs(step[i]) >= 0.8:
            y = max(c0[i], c1[i]) + 0.35 if step[i] >= 0 else min(c0[i], c1[i]) - 0.35
            ax.text(
                xc[i],
                y,
                f"{step[i]:+.1f}",
                ha="center",
                va="bottom" if step[i] >= 0 else "top",
                fontsize=8.5,
                color=col,
                fontweight="bold",
            )
        nxt = x0[i] + w[i]
        ax.plot(
            [nxt, nxt + (T_GAP if i == len(d) - 1 else 0)],
            [c1[i], c1[i]],
            color=T_MUTED,
            lw=0.8,
            ls=":",
        )
    ox0 = 100 + T_GAP
    ax.bar(
        ox0, ov, width=T_OVR_W, align="edge", color=T_BLUE_DK if ov >= 0 else T_RED_DK
    )
    # "even" marker: the bar first climbs back from the 2024 margin to a tie, the rest is the 2026 margin
    even = -ov24
    ax.text(
        ox0 + T_OVR_W / 2,
        max(ov, even) + 0.5,
        f"{ov:+.1f}",
        ha="center",
        va="bottom",
        fontsize=15,
        fontweight="bold",
        color=T_BLUE_DK if ov >= 0 else T_RED_DK,
    )
    ax.plot(
        [ox0 - 1.4, ox0 + T_OVR_W + 1.4],
        [even, even],
        color=T_GREEN,
        lw=2.8,
        solid_capstyle="butt",
        zorder=5,
    )
    ax.text(
        ox0 + T_OVR_W + 1.8,
        even,
        "EVEN",
        ha="left",
        va="center",
        fontsize=8.5,
        fontweight="bold",
        color="black",
        path_effects=[pe.withStroke(linewidth=3, foreground="white")],
        zorder=7,
    )
    margin_lab = tfmt1(ov26)
    if ov26 >= 0:  # bar passes even: shade and label the winning margin
        ax.bar(
            ox0,
            ov - even,
            width=T_OVR_W,
            bottom=even,
            align="edge",
            color=T_BLUE,
            zorder=4,
        )
        thin = (ov - even) < 0.08 * (max(ov, even) - min(0, ov))
        if (
            thin
        ):  # margin segment too thin to hold a label: put it just under the EVEN line
            ax.annotate(
                margin_lab,
                xy=(ox0 + T_OVR_W / 2, even),
                xytext=(0, -6),
                textcoords="offset points",
                ha="center",
                va="top",
                fontsize=10,
                fontweight="bold",
                color="white",
                zorder=6,
            )
        else:
            ax.text(
                ox0 + T_OVR_W / 2,
                (even + ov) / 2,
                margin_lab,
                ha="center",
                va="center",
                fontsize=10,
                fontweight="bold",
                color="white",
                zorder=6,
            )
    else:  # bar falls short of even: bracket the gap above the bar
        ax.annotate(
            "",
            xy=(ox0 + T_OVR_W / 2, even),
            xytext=(ox0 + T_OVR_W / 2, ov),
            arrowprops=dict(arrowstyle="-", color=T_RED, lw=1.2, ls=(0, (2, 1.5))),
        )
        ax.text(
            ox0 + T_OVR_W / 2,
            ov - 0.4,
            f"{margin_lab}\nshort",
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold",
            color="white",
            linespacing=1.15,
            zorder=6,
        )

    # each group's own swing, with its 2026 margin in parens, in a row under the waterfall
    prev_drop, prev_w = False, 0.0
    for i in range(len(d)):
        lab = "0" if round(shift[i]) == 0 else f"{shift[i]:+.0f}"
        narrow = w[i] < 5
        ax.annotate(
            lab,
            xy=(xc[i], 0),
            xycoords=("data", "axes fraction"),
            xytext=(0, -5),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=6.5 if narrow else 8.5,
            fontweight="bold",
            color=T_BLUE if shift[i] >= 0 else T_RED,
        )
        tight = (
            w[i] < 6.5
        )  # narrow bars: drop the parens and shrink so neighbours don't collide
        m26 = tfmt(d.margin26.iloc[i])
        mtxt, mfs = (m26 if tight else f"({m26})"), (5.8 if tight else 7.5)
        clash = (
            tight
            and i > 0
            and not prev_drop
            and (t_text_pt(mtxt, mfs) + prev_w) / 2
            > (xc[i] - xc[i - 1]) * T_PT_PER_X - 2
        )
        prev_drop, prev_w = clash, t_text_pt(mtxt, mfs)
        ax.annotate(
            mtxt,
            xy=(xc[i], 0),
            xycoords=("data", "axes fraction"),
            xytext=(0, (-15 if narrow else -16.5) - (7 if clash else 0)),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=mfs,
            color=T_MUTED,
        )
    ax.annotate(
        "← each group's own swing\n(its 2026 margin)",
        xy=(ox0, 0),
        xycoords=("data", "axes fraction"),
        xytext=(0, -5),
        textcoords="offset points",
        ha="left",
        va="top",
        fontsize=7.5,
        color=T_MUTED,
        linespacing=1.15,
    )
    ax.axhline(0, color=T_INK, lw=0.9)
    ext = [ov, even]
    lo = min(0, c0.min(), c1.min(), *ext)
    hi = max(0, c0.max(), c1.max(), *ext)
    pad = (hi - lo) * 0.12 + 1
    ax.set_ylim(lo - (pad if lo < 0 else 0.6), hi + pad)
    ax.set_xlim(-1, T_XMAX + 1)
    ax.set_ylabel("points added to the D − R margin", fontsize=9.5, color=T_MUTED)
    ax.grid(axis="y", color=T_GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=T_MUTED, labelsize=9)
    ax.spines["left"].set_color(T_GRID)
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(axis="x", labelbottom=False, length=0)

    # ---------- share-of-voters strip (same x axis) ----------
    bx = fig.add_axes([0.10, 0.185, 0.86, 0.21], sharex=ax)
    s24, s26 = d.share24.values, d.share26.values
    smax = max(s24.max(), s26.max())
    smin = min(s24.min(), s26.min())
    span = smax - smin
    bx.set_ylim(smin - 0.25 * span - 7, smax + 5)
    y_lo, y_hi = bx.get_ylim()
    pt_per_y = (0.21 * 8.9 * 72) / (y_hi - y_lo)  # points per y data unit
    placed = []  # label boxes already drawn: (x0, x1, y0, y1) in points
    for i in range(len(d)):
        bx.plot(
            [x0[i] + 0.4, x0[i] + w[i] - 0.4],
            [s26[i]] * 2,
            color=T_BLUE_DK,
            lw=2.4,
            solid_capstyle="butt",
        )
        bx.plot(
            [x0[i] + 0.4, x0[i] + w[i] - 0.4],
            [s24[i]] * 2,
            color=T_LIGHT,
            lw=1.8,
            ls=(0, (2, 1.5)),
        )
        # label each line directly: 2026 share in black, 2024 share in light-blue italic; the higher line's label goes above
        up26 = s26[i] >= s24[i]
        small = w[i] < 6.5
        for val, is26 in ((s26[i], True), (s24[i], False)):
            above = up26 if is26 else not up26
            txt = tpct(val)
            fs = (6.2 if small else 8.5) if is26 else (5.8 if small else 7.8)
            # only nudge a label further from its line if it would actually collide with a neighbour's label
            hw = t_text_pt(txt, fs) / 2
            cx_pt = xc[i] * T_PT_PER_X
            h = fs * 1.15
            off = 4.0
            for _ in range(3):
                base = (val - y_lo) * pt_per_y
                box = (
                    (cx_pt - hw, cx_pt + hw, base + off, base + off + h)
                    if above
                    else (cx_pt - hw, cx_pt + hw, base - off - h, base - off)
                )
                hit = [
                    b
                    for b in placed
                    if b[0] < box[1]
                    and box[0] < b[1]
                    and b[2] < box[3]
                    and box[2] < b[3]
                ]
                if not hit:
                    break
                off += (
                    min(b[3] for b in hit) - box[2]
                    if above
                    else box[3] - max(b[2] for b in hit)
                ) + 1
            placed.append(box)
            bx.annotate(
                txt,
                xy=(xc[i], val),
                xytext=(0, off if above else -off),
                textcoords="offset points",
                ha="center",
                va="bottom" if above else "top",
                fontsize=fs,
                fontweight="bold" if is26 else "normal",
                fontstyle="normal" if is26 else "italic",
                color=T_INK if is26 else T_LIGHT,
                path_effects=T_HALO,
            )
    bx.set_yticks([])
    bx.spines["left"].set_visible(False)

    # cumulative-share underlay on its own 0-100% scale: shaded = 2026, dotted = 2024
    cx = bx.twinx()
    edges = np.concatenate([[0], np.cumsum(w)])
    w24n = d.w24.values / d.w24.values.sum() * 100
    cum24 = np.concatenate([[0], np.cumsum(w24n)])
    # step chart: each group's span sits at the cumulative share through that group
    step26 = np.append(edges[1:], edges[-1])
    step24 = np.append(cum24[1:], cum24[-1])
    cx.fill_between(edges, 0, step26, step="post", color=T_BLUE, alpha=0.10, lw=0)
    cx.plot(edges, step26, drawstyle="steps-post", color=T_BLUE, alpha=0.5, lw=1.1)
    cx.plot(
        edges,
        step24,
        drawstyle="steps-post",
        color=T_GREY,
        alpha=0.9,
        lw=1.2,
        ls=(0, (1.5, 1.5)),
    )
    cx.set_ylim(0, 104)
    cx.set_yticks([])
    cx.spines["right"].set_visible(False)
    cx.spines["left"].set_visible(False)
    for age in dict.fromkeys(g.split(" ", 1)[1] for g in d.group):
        j = max(i for i, g in enumerate(d.group) if g.endswith(age)) + 1
        cx.text(
            edges[j] - 0.7,
            2.5,
            f"{edges[j]:.1f}%",
            ha="right",
            va="bottom",
            fontsize=7.5,
            color=T_MUTED,
            path_effects=T_HALO,
        )

    bx.set_zorder(cx.get_zorder() + 1)
    bx.patch.set_visible(False)

    # strip legend (x in data units, y in axes fraction)
    tr = bx.get_xaxis_transform()
    lx = ox0
    bx.text(
        lx,
        1.02,
        "Who showed up",
        transform=tr,
        fontsize=9.5,
        color=T_INK,
        fontweight="bold",
        va="top",
    )
    bx.text(
        lx, 0.90, "share of voters", transform=tr, fontsize=7.5, color=T_MUTED, va="top"
    )
    bx.plot([lx, lx + 4], [0.74, 0.74], transform=tr, color=T_BLUE_DK, lw=2.4)
    bx.text(
        lx + 5,
        0.74,
        "2026",
        transform=tr,
        va="center",
        fontsize=8,
        color=T_INK,
        fontweight="bold",
    )
    bx.plot(
        [lx, lx + 4],
        [0.63, 0.63],
        transform=tr,
        color=T_LIGHT,
        lw=1.8,
        ls=(0, (2, 1.5)),
    )
    bx.text(
        lx + 5,
        0.63,
        "2024",
        transform=tr,
        va="center",
        fontsize=8,
        color=T_LIGHT,
        fontstyle="italic",
    )
    bx.add_patch(
        Rectangle((lx, 0.47), 4, 0.08, transform=tr, color=T_BLUE, alpha=0.18, lw=0)
    )
    bx.text(
        lx + 5,
        0.51,
        "cumulative,\nto 100%",
        transform=tr,
        va="center",
        fontsize=7,
        color=T_INK,
        linespacing=1.1,
    )
    bx.plot(
        [lx, lx + 4],
        [0.36, 0.36],
        transform=tr,
        color=T_GREY,
        alpha=0.75,
        lw=1.0,
        ls=(0, (1.5, 1.5)),
    )
    bx.text(
        lx + 5,
        0.36,
        "2024\ncumulative",
        transform=tr,
        va="center",
        fontsize=7,
        color=T_INK,
        linespacing=1.1,
    )

    # x labels: M / W under each bar, age band under each pair
    bx.set_xticks(xc)
    bx.set_xticklabels(
        ["M" if not wm else "W" for wm in women], fontsize=8.5, color=T_MUTED
    )
    bx.tick_params(axis="x", length=0, pad=3)
    bx.spines["bottom"].set_color(T_GRID)
    for age in dict.fromkeys(g.split(" ", 1)[1] for g in d.group):
        idx = [i for i, g in enumerate(d.group) if g.endswith(age)]
        xm = (x0[idx[0]] + x0[idx[-1]] + w[idx[-1]]) / 2
        bx.annotate(
            age.replace("-", "–"),
            xy=(xm, 0),
            xycoords=("data", "axes fraction"),
            xytext=(0, -22),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=10,
            fontweight="bold",
            color=T_INK,
        )

    # ---------- footer ----------
    fig.text(
        0.06,
        0.018,
        f"Sources: 2026 SSRS Voter Poll (network exit poll, via Fox News) · 2024 {source_2024}, Harris–Trump · actual counts: Fox News, Nov 7.\n"
        "Group margins are exit-poll estimates adjusted to match the actual vote: each year's gap between the poll and the count is\n"
        "spread evenly across all voters (every voter counts the same). Group shares of the electorate are exit-poll estimates.\n"
        "Bar height = change in the group's net vote as a share of all voters (2026 share × margin − 2024 share × margin). Bars sum to the swing.\n"
        "Green line = EVEN: below it the swing erases the 2024 margin; above it is the 2026 margin. Margin = Dem − Rep. 2026 counts may not be final.",
        fontsize=7.4,
        color=T_MUTED,
        va="bottom",
        linespacing=1.35,
    )
    fig.savefig(path, dpi=200, facecolor="white")
    plt.close(fig)


logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
actual_idx = pd.read_csv("data/actual_results.csv").set_index(["office", "state"])
out_dir = pathlib.Path("thumbnails")
out_dir.mkdir(exist_ok=True)
SRC24 = {
    "Edison 2024": "Edison exit poll (via CNN)",
    "VoteCast 2024": "AP VoteCast (via Fox News)",
}
for (office, st), g in thumb_sel.groupby(["office", "state"]):
    make_thumb(
        g,
        office,
        st,
        out_dir / f"{st}_{office.lower()}_2026.png",
        actual=actual_idx.loc[(office, st)],
        source_2024=SRC24[g.baseline.iloc[0]],
    )
print(
    f"wrote {thumb_sel.groupby(['office', 'state']).ngroups} thumbnails to {out_dir}/"
)

wrote 16 thumbnails to thumbnails/
